# Gold Layer: Wind Turbine Daily Aggregates

Reads from `silver.wind_turbine_clean`, computes per-turbine daily summary statistics

## Output tables

| Table | Grain | Contents |
|---|---|---|
| `wind_turbine_daily_stats` | turbine × day | min / max / avg power output |

In [ ]:
# ---------------------------------------------------------------------------
# Parameters (overridden by job/widget at runtime)
# ---------------------------------------------------------------------------
dbutils.widgets.text("catalog", "wind_farm_dev")
dbutils.widgets.text("schema",  "wind_farm")

catalog = dbutils.widgets.get("catalog")
schema  = dbutils.widgets.get("schema")

silver_table = f"`{catalog}`.`{schema}`.wind_turbine_clean"
stats_table  = f"`{catalog}`.`{schema}`.wind_turbine_daily_stats"

print(f"Source         : {silver_table}")
print(f"Stats target   : {stats_table}")

In [ ]:
from pyspark.sql import functions as F

silver_df = spark.table(silver_table)
print(f"Silver row count: {silver_df.count():,}")

## Step 1 — Per-turbine daily summary statistics

In [ ]:
from wind_farm.utils.turbine import TurbineAggregator

daily_stats = TurbineAggregator.daily_aggregate(silver_df)

display(daily_stats)

## Step 2 — Write gold Delta tables

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")

(
    daily_stats.write
        .format("delta")
        .mode("overwrite") # Overwrite gold table
        .option("overwriteSchema", "true")
        .saveAsTable(stats_table)
)
print(f"Stats write complete: {stats_table}")


## Step 3 — Summary

In [ ]:
stats_written   = spark.table(stats_table).count()

date_range = (
    spark.table(stats_table)
        .agg(F.min("date").alias("from"), F.max("date").alias("to"))
        .collect()[0]
)

print("=" * 55)
print(f"Date range covered: {date_range['from']} → {date_range['to']}")
print(f"Turbine-day stats: {stats_written:>8,}")
print("=" * 55)